# NebulaGraph as Graph Memory

## Prerequisites

### 1. Install Mem0 with Graph Memory support 

To use Mem0 with Graph Memory support, install it using pip:

```bash
pip install "mem0ai[graph]"
```

This command installs Mem0 along with the necessary dependencies for graph functionality.

### 2. Install NebulaGraph (docker-compose)

NebulaGraph is a distributed graph database that consists of three services that work together:
- **graphd**: query service
- **metad**: metadata service
- **storaged**: storage service

`graphd` depends on `metad` and `storaged`. Use the official docker-compose deployment for local development.

```bash
git clone -b v3.8 https://github.com/vesoft-inc/nebula-docker-compose.git
cd nebula-docker-compose
docker compose up -d
```

### 3. Create Graph Space

Connect to NebulaGraph and create a graph space:

```bash
docker exec \
  -it nebula-docker-compose-console-1 \
  nebula-console \
  -addr graphd \
  -port 9669 \
  -u root \
  -p nebula
```

Then execute:

```cypher
CREATE SPACE IF NOT EXISTS mem0_space( \
  partition_num=9, \
  replica_factor=3, \
  vid_type=FIXED_STRING(36) \
);
USE mem0_space;
```

### 4. Install Elasticsearch (for vector search)

NebulaGraph v3.8 doesn't support native vector search, so we use Elasticsearch:

```bash
docker run -d \
  --name elasticsearch \
  -p 9200:9200 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  docker.elastic.co/elasticsearch/elasticsearch:8.15.0
```

### 5. Install Other Dependencies
```bash
pip install nebula3-python rank-bm25 "elasticsearch>=8.15,<9"
```

Additional information can be found on [NebulaGraph documentation](https://docs.nebula-graph.io/).


## Configuration

Do all the imports and configure OpenAI (enter your OpenAI API key):

In [ ]:
from mem0 import Memory

import os

os.environ["OPENAI_API_KEY"] = ""

Set up configuration to use the embedder model, NebulaGraph as a graph store, and Elasticsearch for vector search:

In [ ]:
config = {
    "embedder": {
        "provider": "openai",
        "config": {"model": "text-embedding-3-large", "embedding_dims": 1536},
    },
    "graph_store": {
        "provider": "nebulagraph",
        "config": {
            "graph_address": "localhost:9669",
            "username": "root",
            "password": "nebula",
            "space": "mem0_space",
            "delete_page_size": 1000, # Batch size for delete_all
            "collection_name": "mem0_nebulagraph_vectors_graph" # The entity vectors are stored separately from the vector store.
        },
    },
    "vector_store": {
        "provider": "elasticsearch",
        "config": {
            "host": "http://localhost",
            "port": 9200,
            "collection_name": "mem0_nebulagraph_vectors",
        },
    },
}

## Graph Memory initialization 

Initialize NebulaGraph as a Graph Memory store: 

In [ ]:
m = Memory.from_config(config_dict=config)

## Store memories 

Create memories:

In [ ]:
messages = [
    {
        "role": "user",
        "content": "I'm planning to watch a movie tonight. Any recommendations?",
    },
    {
        "role": "assistant",
        "content": "How about a thriller movies? They can be quite engaging.",
    },
    {
        "role": "user",
        "content": "I'm not a big fan of thriller movies but I love sci-fi movies.",
    },
    {
        "role": "assistant",
        "content": "Got it! I'll avoid thriller recommendations and suggest sci-fi movies in the future.",
    },
]

Store memories in NebulaGraph:

In [ ]:
# Store inferred memories (default behavior)
result = m.add(messages, user_id="alice", metadata={"category": "movie_recommendations"})

## Search memories

In [ ]:
for result in m.search("what does alice love?", user_id="alice")["results"]:
    print(result["memory"], result["score"])

Prefers sci-fi movies over thriller movies 0.70897293
Planning to watch a movie tonight 0.70391846


## Get all memories

In [ ]:
all_memories = m.get_all(user_id="alice")
for memory in all_memories['relations']:
    print(f"{memory['source']} -> {memory['relationship']} -> {memory['target']}")

alice -> prefer -> sci-fi
alice -> ask_for_recommendations -> movie
alice -> planning_to_watch -> movie
alice -> dislike -> thriller
